# Préparation des volumes CT pour les VLM

**Entrée :** un volume `.nii.gz` (T00, T50 ou un recalage).
**Sortie :** un dossier avec
- `coupes_16/slice_01.png ... slice_16.png` : coupes axiales dans l'ordre, du haut (tête) vers le bas
- `mosaique_3.png` : les coupes à 30, 50 et 70 % côte à côte
- `manifeste.json` : quelle coupe du volume correspond à chaque PNG

Les choix suivent la recette du README de Vincent (`/home/student/vqa/README_STARTUP.md`, section 6).
Ce notebook ne fait **que** préparer les images. L'envoi au VLM (avec le prompt et les tags `SLICE i`) est une autre étape.

**Dans Colab :** glisse tes fichiers `.nii.gz` dans le panneau Fichiers (icône dossier, à gauche).
Attention, Colab efface les fichiers à la fin de la session : télécharge le zip à la fin (dernière cellule).

In [ ]:
!pip install -q nibabel

import json
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt

## 1. Paramètres

Tout ce qu'on peut vouloir changer est ici, et nulle part ailleurs.

| Paramètre | Valeur | D'où ça vient |
|---|---|---|
| `N_COUPES` | 16 | README 6.2 : jusqu'à 85 coupes ; README 6.4 : « commencer avec 8-16, pas 85 » |
| `TAILLE` | 896 | README 6.2 étape 4 : chaque coupe en 896×896 |
| `FENETRES` | 3 fenêtres HU | README 6.2 étape 5 : une fenêtre par canal R, G, B |
| `MOSAIQUE_POSITIONS` | 30, 50, 70 % | Le test d'Elias (le mosaïque a donné de meilleurs résultats) |

In [ ]:
VOLUME_PATH = "case1_T00_native.nii.gz"   # le fichier à traiter
OUTPUT_DIR = "sorties"                    # dossier de sortie

N_COUPES = 16                             # nombre de coupes axiales
TAILLE = 896                              # taille d'une coupe en pixels
MOSAIQUE_POSITIONS = (0.30, 0.50, 0.70)   # en % depuis le haut du volume

# Fenêtres HU -> canaux R, G, B (README section 6.2, étape 5)
FENETRES = [
    (-1024, 1024),   # R : large, de l'air/poumon jusqu'à l'os
    (-135, 215),     # G : tissus mous
    (0, 80),         # B : étroite, faite pour le cerveau
]

## 2. Lire le volume et le remettre dans le bon sens

**Pourquoi :** on a vu sur ITK-SNAP que les fichiers n'ont pas tous la même orientation dans leur en-tête (LPS, RAI, RAS...).
Le README demande de mettre le volume en axial (étape 1). `nib.as_closest_canonical` remet **tous** les volumes dans le même sens (RAS),
quelle que soit leur orientation d'origine. Comme ça, la coupe 1 est toujours en haut et l'image est toujours dans le même sens.

**Limite :** la fonction fait confiance à l'en-tête. Si l'en-tête est faux (comme le fichier en RAI qu'on a vu), le résultat sera à l'envers.
C'est pour ça qu'on vérifie les images à la main à l'étape 6.

In [ ]:
def charger_volume(chemin):
    img = nib.load(chemin)
    orientation = "".join(nib.aff2axcodes(img.affine))   # convention Python/nibabel (ex. LPI)
    img_ras = nib.as_closest_canonical(img)               # tout le monde en RAS
    vol = np.asanyarray(img_ras.dataobj).astype(np.float32)
    spacing = tuple(float(s) for s in img_ras.header.get_zooms()[:3])
    return vol, orientation, spacing


vol, orientation, spacing = charger_volume(VOLUME_PATH)
print("Orientation dans le fichier :", orientation)
print("Taille :", vol.shape, "| spacing (mm) :", spacing)
print("Intensités : min", vol.min(), "| max", vol.max())

**Ce que tu dois voir :** une taille du genre `(256, 256, 94)` et un minimum autour de **-1024** (l'air en HU).
Si le minimum est autour de 0, le volume n'est pas en HU et le fenêtrage ne marchera pas : il faut le signaler.

## 3. Choisir les coupes

**Pourquoi :** le README dit de prendre les coupes **à intervalles réguliers** le long de z (étape 3), pas à la main.
On couvre tout le volume, et n'importe qui obtient les mêmes coupes (c'est reproductible).
C'est exactement la formule du README : `np.linspace(0, d - 1, n).round()`.

On les range **du haut (tête) vers le bas**, comme quand on fait défiler un scanner.

In [ ]:
def indices_equidistants(nb_total, n):
    n = min(n, nb_total)
    idx = np.linspace(0, nb_total - 1, n).round().astype(int)
    return idx[::-1]   # en RAS, z augmente vers la tête : on inverse pour aller du haut vers le bas


def indices_mosaique(nb_total, positions):
    # position 0 = tout en haut, 1 = tout en bas
    return [int(round((1 - p) * (nb_total - 1))) for p in positions]


nz = vol.shape[2]
print("Coupes choisies (index z) :", indices_equidistants(nz, N_COUPES))
print("Coupes du mosaïque       :", indices_mosaique(nz, MOSAIQUE_POSITIONS))

## 4. Fenêtrage en couleur

**Pourquoi :** c'est le cœur de la recette (README 6.2, étape 5). Au lieu d'une seule fenêtre de contraste,
on en fait **trois**, et on met chacune dans une couleur. Le modèle reçoit trois « vues » de la même coupe dans une seule image.
Le README précise qu'Ollama ne le fait pas tout seul : il faut le faire avant d'envoyer.

**Orientation de l'affichage :** on met l'avant du patient en haut et sa droite à gauche de l'écran
(convention radiologique, la même qu'ITK-SNAP). Les coupes sont carrées (spacing 0,97 × 0,97 mm), donc l'agrandissement à 896 ne déforme rien.

In [ ]:
def coupe_axiale(vol, k):
    # En RAS : axe 0 = gauche -> droite, axe 1 = arrière -> avant.
    # .T met l'avant/arrière en lignes ; [::-1, ::-1] met l'avant en haut et la droite du patient à gauche.
    return vol[:, :, k].T[::-1, ::-1]


def fenetrage_rgb(coupe_hu, fenetres=FENETRES, taille=TAILLE):
    canaux = []
    for bas, haut in fenetres:
        x = np.clip(coupe_hu, bas, haut)
        x = (x - bas) / (haut - bas) * 255.0
        canaux.append(x.astype(np.uint8))
    rgb = np.stack(canaux, axis=-1)
    return Image.fromarray(rgb, "RGB").resize((taille, taille))   # même appel que le README

## 5. Tout assembler et enregistrer

Une seule fonction fait tout pour un volume. C'est elle qu'on réutilisera pour les 50 volumes.
On enregistre en **PNG** (sans perte, comme demandé dans le vademecum).

In [ ]:
def preparer_volume(chemin, dossier_sortie=OUTPUT_DIR, n_coupes=N_COUPES):
    vol, orientation, spacing = charger_volume(chemin)
    nz = vol.shape[2]
    nom = Path(chemin).name.replace(".nii.gz", "").replace(".nii", "")
    sortie = Path(dossier_sortie) / nom
    dossier_coupes = sortie / f"coupes_{n_coupes}"
    dossier_coupes.mkdir(parents=True, exist_ok=True)

    # Coupes ordonnées
    coupes = []
    for rang, k in enumerate(indices_equidistants(nz, n_coupes), start=1):
        fichier = f"slice_{rang:02d}.png"
        fenetrage_rgb(coupe_axiale(vol, k)).save(dossier_coupes / fichier)
        coupes.append({"slice": rang, "fichier": fichier, "z_voxel": int(k),
                       "position_depuis_haut_pct": round(100 * (1 - k / (nz - 1)), 1)})

    # Mosaïque de 3
    tuiles = [fenetrage_rgb(coupe_axiale(vol, k)) for k in indices_mosaique(nz, MOSAIQUE_POSITIONS)]
    mosaique = Image.new("RGB", (TAILLE * len(tuiles), TAILLE))
    for i, tuile in enumerate(tuiles):
        mosaique.paste(tuile, (i * TAILLE, 0))
    mosaique.save(sortie / "mosaique_3.png")

    manifeste = {
        "volume": Path(chemin).name,
        "orientation_fichier": orientation,
        "taille": list(vol.shape),
        "spacing_mm": spacing,
        "n_coupes": len(coupes),
        "taille_png": TAILLE,
        "fenetres_hu_rgb": FENETRES,
        "mosaique_positions": list(MOSAIQUE_POSITIONS),
        "coupes": coupes,
    }
    (sortie / "manifeste.json").write_text(json.dumps(manifeste, indent=2))
    return sortie


sortie = preparer_volume(VOLUME_PATH)
print("Écrit dans :", sortie)

## 6. Vérifier avec les yeux

C'est l'étape la plus importante. Le code ne peut pas savoir si une image est à l'envers : toi, oui.

**Liste de contrôle :**
1. **Couleurs :** les images sont rougeâtres/jaunâtres, pas grises. C'est normal (les trois fenêtres).
2. **Ordre :** `slice_01` = le haut du thorax (sommet des poumons), `slice_16` = le bas (diaphragme, foie).
3. **Sens :** l'avant du patient (sternum) en **haut**, le dos (colonne) en **bas**.
4. **Gauche/droite :** le cœur penche vers la **droite de l'écran** (= gauche du patient, convention radiologique).
5. **Rien de coupé ni d'écrasé :** le thorax entier, bien rond.

In [ ]:
def afficher_coupes(dossier):
    fichiers = sorted(Path(dossier).glob("slice_*.png"))
    colonnes = 4
    lignes = int(np.ceil(len(fichiers) / colonnes))
    fig, axes = plt.subplots(lignes, colonnes, figsize=(3 * colonnes, 3 * lignes))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, f in zip(np.ravel(axes), fichiers):
        ax.imshow(Image.open(f))
        ax.set_title(f.stem)
    plt.tight_layout()
    plt.show()


afficher_coupes(sortie / f"coupes_{N_COUPES}")

In [ ]:
# Le mosaïque
plt.figure(figsize=(15, 5))
plt.imshow(Image.open(sortie / "mosaique_3.png"))
plt.title("mosaique_3 : 30 % | 50 % | 70 %")
plt.axis("off")
plt.show()

### Pour comprendre les trois couleurs

On sépare une coupe du milieu en ses trois canaux. Tu verras que :
- **R** (large) montre l'intérieur des poumons ;
- **G** (tissus mous) montre bien le cœur et les muscles, poumons noirs ;
- **B** (étroite) est presque binaire : c'est une fenêtre faite pour le cerveau.

In [ ]:
k_milieu = indices_equidistants(nz, N_COUPES)[len(indices_equidistants(nz, N_COUPES)) // 2]
rgb = np.array(fenetrage_rgb(coupe_axiale(vol, k_milieu)))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(rgb); axes[0].set_title("RGB (ce que voit le VLM)")
for i, nom in enumerate(["R : -1024 / 1024", "G : -135 / 215", "B : 0 / 80"]):
    axes[i + 1].imshow(rgb[:, :, i], cmap="gray")
    axes[i + 1].set_title(nom)
for ax in axes:
    ax.axis("off")
plt.show()

## 7. Plusieurs volumes d'un coup

Mets ici la liste des fichiers que tu as glissés dans Colab. Pour les tests : les cas 1, 4 et 9.

In [ ]:
VOLUMES = [
    "case1_T00_native.nii.gz",
    # "Case4_T00.nii.gz",
    # "01_baseline_case4.nii.gz",
]

for chemin in VOLUMES:
    if Path(chemin).exists():
        print("OK  ", chemin, "->", preparer_volume(chemin))
    else:
        print("ABSENT", chemin)

## 8. Télécharger les résultats

Colab efface tout à la fin de la session. Cette cellule fait un zip et le télécharge.

In [ ]:
shutil.make_archive("sorties", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download("sorties.zip")
except ImportError:
    print("Pas dans Colab : le fichier sorties.zip est dans le dossier courant.")